# Clustering Analysis — Whisper Encoder Representations
Visualisation and analysis of phoneme/L1/speaker clustering metrics across encoder layers.


In [1]:
import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import display, Image, HTML
from pathlib import Path
import ipywidgets as widgets

# ── Config ────────────────────────────────────────────────────────────────────
RESULTS_JSON = 'results/clustering/clustering_scripted.json'
PLOTS_DIR    = Path('results/clustering')
LAYERS       = list(range(7))
MODELS       = ['baseline', 'lora']
LABEL_TYPES  = ['phoneme', 'l1', 'speaker']

with open(RESULTS_JSON) as f:
    data = json.load(f)

# ── Flatten to DataFrame ──────────────────────────────────────────────────────
rows = []
for model in MODELS:
    for layer in LAYERS:
        for label_type in LABEL_TYPES:
            d = data.get(model, {}).get(str(layer), {}).get(label_type, {})
            if 'error' in d: continue
            rows.append({
                'model': model, 'layer': layer, 'label_type': label_type,
                'silhouette':          d.get('silhouette'),
                'davies_bouldin':      d.get('davies_bouldin'),
                'calinski_harabasz':   d.get('calinski_harabasz'),
                'wb_ratio':            d.get('wb_ratio'),
                'within_mean':         d.get('within_mean'),
                'between_mean':        d.get('between_mean'),
                'n_samples':           d.get('n_samples'),
                'n_classes':           d.get('n_classes'),
            })

df = pd.DataFrame(rows)
print(f'Loaded {len(df)} metric rows')
display(df.head())

Loaded 42 metric rows


,model,layer,label_type,silhouette,davies_bouldin,calinski_harabasz,wb_ratio,within_mean,between_mean,n_samples,n_classes
0,baseline,0,phoneme,-0.095716,9.543646,9.996934,2.257227,25.538538,11.314119,201520,39
1,baseline,0,l1,0.005427,9.086597,39.987845,3.277779,26.777723,8.169473,201520,6
2,baseline,0,speaker,0.002672,8.943296,40.653765,3.240796,26.794823,8.267977,201520,6
3,baseline,1,phoneme,-0.050166,7.373102,14.756587,1.853670,25.116703,13.549716,201520,39
4,baseline,1,l1,0.023226,7.514719,55.668627,2.769856,26.660490,9.625225,201520,6


---
# Section 1 — UMAP Plots by Label Type (All Layers per Model)

In [3]:
def show_umap_grid(model, plot_type, umap_layers=[0, 3, 6]):
    """Display UMAP plots for a given model and plot type in a row."""
    print(f'\n── {model.upper()} | {plot_type} ──')
    imgs = []
    for layer in umap_layers:
        path = PLOTS_DIR / model / f'umap_{model}_layer{layer}_{plot_type}.png'
        if path.exists():
            imgs.append(widgets.Image(value=open(path,'rb').read(), format='png',
                                      layout=widgets.Layout(width='32%')))
        else:
            imgs.append(widgets.Label(f'Missing: {path.name}'))
    display(widgets.HBox(imgs,
        layout=widgets.Layout(justify_content='space-around')))
    display(HTML(f'<p style="text-align:center;color:#888">Layers: {umap_layers}</p>'))

display(HTML('<h3>L1 / Accent Clusters</h3>'))
for model in MODELS:
    show_umap_grid(model, 'l1')

display(HTML('<h3>Phoneme Clusters</h3>'))
for model in MODELS:
    show_umap_grid(model, 'phoneme')

display(HTML('<h3>Speaker Clusters</h3>'))
for model in MODELS:
    show_umap_grid(model, 'speaker')


── BASELINE | l1 ──



── LORA | l1 ──



── BASELINE | phoneme ──



── LORA | phoneme ──



── BASELINE | speaker ──



── LORA | speaker ──


---
# Section 2 — Side-by-Side Layer Comparisons (Baseline vs LoRA)

In [4]:
umap_layers = [0, 3, 6]
for plot_type in ['l1', 'phoneme', 'focus_phone', 'focus_l1']:
    # Check if any plots exist for this type
    any_exist = any(
        (PLOTS_DIR / m / f'umap_l{l}_{plot_type}.png').exists()
        for m in MODELS for l in umap_layers
    )
    if not any_exist: continue

    display(HTML(f'<h3>{plot_type.replace("_"," ").title()} — Baseline vs LoRA by Layer</h3>'))
    for layer in umap_layers:
        row_imgs = []
        for model in MODELS:
            path = PLOTS_DIR / model / f'umap_l{layer}_{plot_type}.png'
            label = f'{model} | Layer {layer}'
            if path.exists():
                img = widgets.Image(value=open(path,'rb').read(), format='png',
                                    layout=widgets.Layout(width='48%'))
                box = widgets.VBox([
                    widgets.HTML(f'<b style="font-size:13px">{label}</b>'), img
                ], layout=widgets.Layout(align_items='center', width='49%'))
            else:
                box = widgets.Label(f'Missing: {path.name}')
            row_imgs.append(box)
        display(widgets.HBox(row_imgs,
            layout=widgets.Layout(justify_content='space-around', margin='8px 0')))
    display(HTML('<hr style="border:0.5px solid #ddd"/>'))

---
# Section 3 — Metric Tables and Layer-by-Layer Progression

In [5]:
# ── 3a. Pivot tables: one metric, all layers × models ─────────────────────────
METRICS = {
    'silhouette':        ('Silhouette ↑',        'higher is better'),
    'davies_bouldin':    ('Davies-Bouldin ↓',     'lower is better'),
    'calinski_harabasz': ('Calinski-Harabasz ↑',  'higher is better'),
    'wb_ratio':          ('Within/Between Ratio ↓','lower is better'),
}

for label_type in LABEL_TYPES:
    display(HTML(f'<h4>Label: <code>{label_type}</code></h4>'))
    sub = df[df['label_type'] == label_type].copy()
    for metric, (title, note) in METRICS.items():
        pivot = sub.pivot(index='layer', columns='model', values=metric)
        pivot.columns.name = None
        pivot.index.name = 'Layer'
        # Highlight best per row
        display(HTML(f'<b>{title}</b> <span style="color:#888;font-size:12px">({note})</span>'))
        display(pivot.style
            .format('{:.4f}')
            .background_gradient(cmap='RdYlGn' if '↑' in title else 'RdYlGn_r', axis=None)
            .set_caption(f'{label_type} — {title}')
        )
    display(HTML('<hr/>'))

,baseline,lora
Layer,,
0,-0.0957,-0.1231
1,-0.0502,-0.0464
2,-0.0370,-0.0347
3,-0.0226,-0.0421
4,-0.0435,-0.0042
5,-0.0001,0.0043
6,0.0070,0.0061


,baseline,lora
Layer,,
0,9.5436,8.9195
1,7.3731,7.5478
2,7.1780,7.2309
3,6.5317,6.6413
4,6.3525,6.3895
5,5.8317,5.9729
6,5.8608,5.9587


,baseline,lora
Layer,,
0,9.9969,10.4948
1,14.7566,14.6628
2,16.8853,16.4661
3,17.3812,16.7316
4,17.1100,17.2979
5,18.8625,19.7269
6,17.6597,17.7885


,baseline,lora
Layer,,
0,2.2572,2.1778
1,1.8537,1.8813
2,1.7755,1.7721
3,1.6981,1.7354
4,1.7117,1.7378
5,1.6447,1.6351
6,1.6790,1.6670


,baseline,lora
Layer,,
0,0.0054,0.0052
1,0.0232,0.0261
2,0.0288,0.0279
3,0.0227,0.0225
4,0.0212,0.0209
5,0.0199,0.0213
6,0.0232,0.0236


...

,baseline,lora
Layer,,
0,3.2778,3.3458
1,2.7699,2.7374
2,2.9118,2.8940
3,3.2556,3.2380
4,3.5606,3.5180
5,3.6314,3.6637
6,3.6275,3.5975


,baseline,lora
Layer,,
0,0.0027,0.0008
1,0.0253,0.0261
2,0.0287,0.0258
3,0.0257,0.0241
4,0.0219,0.0208
5,0.0195,0.0194
6,0.0220,0.0230


,baseline,lora
Layer,,
0,8.9433,8.7314
1,7.5355,7.3173
2,7.3992,7.4137
3,8.0136,8.0492
4,8.5416,8.3363
5,8.5873,8.5857
6,8.5620,8.5959


,baseline,lora
Layer,,
0,40.6538,41.2618
1,56.1357,57.2547
2,49.4026,51.9774
3,40.2158,41.1602
4,33.2120,33.4995
5,30.9184,31.4248
6,32.5010,32.7153


,baseline,lora
Layer,,
0,3.2408,3.2154
1,2.7650,2.7344
2,2.9334,2.8693
3,3.2323,3.1991
4,3.5622,3.5365
5,3.6679,3.6553
6,3.6127,3.6086


In [6]:
# ── 3b. Line charts: metric progression by layer ──────────────────────────────
COLORS = {'baseline': '#636EFA', 'lora': '#EF553B'}
DASH   = {'baseline': 'solid',   'lora': 'dash'}

for label_type in LABEL_TYPES:
    sub = df[df['label_type'] == label_type]
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=[t for t, _ in METRICS.values()],
    )
    positions = [(1,1),(1,2),(2,1),(2,2)]
    for (metric, (title, note)), (r, c) in zip(METRICS.items(), positions):
        for model in MODELS:
            msub = sub[sub['model'] == model].sort_values('layer')
            fig.add_trace(go.Scatter(
                x=msub['layer'], y=msub[metric],
                name=model, legendgroup=model,
                showlegend=(r==1 and c==1),
                line=dict(color=COLORS[model], dash=DASH[model], width=2),
                mode='lines+markers',
            ), row=r, col=c)
        fig.update_xaxes(title_text='Layer', row=r, col=c)
        fig.update_yaxes(title_text=metric[:12], row=r, col=c)

    fig.update_layout(
        title=f'Clustering Metrics by Layer — Label: {label_type}',
        height=600,
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
    )
    fig.show()

In [7]:
# ── 3c. Cross-label comparison: phoneme vs L1 silhouette on same axes ─────────
fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Baseline', 'LoRA FT'],
    shared_yaxes=True)

LABEL_COLORS = {'phoneme': '#00CC96', 'l1': '#AB63FA', 'speaker': '#FFA15A'}
LABEL_DASH   = {'phoneme': 'solid', 'l1': 'dot', 'speaker': 'dash'}

for ci, model in enumerate(MODELS):
    for label_type in LABEL_TYPES:
        sub = df[(df['model']==model) & (df['label_type']==label_type)].sort_values('layer')
        fig.add_trace(go.Scatter(
            x=sub['layer'], y=sub['silhouette'],
            name=label_type, legendgroup=label_type,
            showlegend=(ci == 0),
            mode='lines+markers',
            line=dict(color=LABEL_COLORS[label_type],
                      dash=LABEL_DASH[label_type], width=2),
        ), row=1, col=ci+1)

fig.add_hline(y=0, line_dash='dot', line_color='gray', line_width=1)
fig.update_xaxes(title_text='Layer')
fig.update_yaxes(title_text='Silhouette Score', col=1)
fig.update_layout(
    title='Silhouette Score by Layer: Phoneme vs L1 vs Speaker',
    height=420,
    legend=dict(orientation='h', yanchor='bottom', y=1.05, xanchor='center', x=0.5),
)
fig.show()

In [8]:
# ── 3d. Baseline vs LoRA delta (LoRA - baseline) per metric ───────────────────
display(HTML('<h4>LoRA − Baseline Delta (positive = LoRA better)</h4>'))
for label_type in LABEL_TYPES:
    sub = df[df['label_type'] == label_type]
    base = sub[sub['model']=='baseline'].set_index('layer')
    lora = sub[sub['model']=='lora'].set_index('layer')
    delta = pd.DataFrame(index=LAYERS)
    for metric, (title, note) in METRICS.items():
        # For 'lower is better' metrics, negate so positive = improvement
        sign = -1 if '↓' in note else 1
        delta[title] = sign * (lora[metric] - base[metric])
    delta.index.name = 'Layer'
    display(HTML(f'<b>{label_type}</b>'))
    display(delta.style
        .format('{:+.4f}')
        .background_gradient(cmap='RdYlGn', axis=None)
        .set_caption(f'LoRA improvement over baseline — {label_type}')
    )

,Silhouette ↑,Davies-Bouldin ↓,Calinski-Harabasz ↑,Within/Between Ratio ↓
Layer,,,,
0,-0.0273,-0.6241,+0.4979,-0.0794
1,+0.0038,+0.1747,-0.0937,+0.0276
2,+0.0023,+0.0529,-0.4193,-0.0034
3,-0.0195,+0.1096,-0.6495,+0.0373
4,+0.0393,+0.0370,+0.1879,+0.0260
5,+0.0044,+0.1412,+0.8644,-0.0096
6,-0.0009,+0.0978,+0.1288,-0.0120


,Silhouette ↑,Davies-Bouldin ↓,Calinski-Harabasz ↑,Within/Between Ratio ↓
Layer,,,,
0,-0.0002,+0.0306,-1.9725,+0.0681
1,+0.0028,-0.1636,+1.2911,-0.0325
2,-0.0009,+0.0442,+0.7020,-0.0178
3,-0.0002,-0.0123,+0.5251,-0.0176
4,-0.0003,-0.1047,+0.7602,-0.0426
5,+0.0014,+0.0302,-0.6397,+0.0323
6,+0.0003,+0.0185,+0.6848,-0.0300


,Silhouette ↑,Davies-Bouldin ↓,Calinski-Harabasz ↑,Within/Between Ratio ↓
Layer,,,,
0,-0.0019,-0.2119,+0.6080,-0.0254
1,+0.0008,-0.2182,+1.1190,-0.0306
2,-0.0029,+0.0144,+2.5749,-0.0641
3,-0.0016,+0.0356,+0.9443,-0.0332
4,-0.0011,-0.2053,+0.2874,-0.0257
5,-0.0001,-0.0016,+0.5064,-0.0125
6,+0.0011,+0.0340,+0.2143,-0.0041


---
# Section 4 — Deeper Diagnostics

In [9]:
# ── 4a. Entanglement index: L1 silhouette / phoneme silhouette ────────────────
# A high ratio means accent is much more structured than phoneme at that layer.
# This is the geometric fingerprint of entanglement.
display(HTML('<h4>Accent-Phoneme Entanglement Index (L1 sil / |phoneme sil|)</h4>'))
display(HTML('<p style="color:#888">Higher = accent more structured than phoneme = more entanglement</p>'))

fig = go.Figure()
for model in MODELS:
    ph  = df[(df['model']==model) & (df['label_type']=='phoneme')].sort_values('layer')
    l1  = df[(df['model']==model) & (df['label_type']=='l1')].sort_values('layer')
    ratio = l1['silhouette'].values / (np.abs(ph['silhouette'].values) + 1e-8)
    fig.add_trace(go.Scatter(
        x=list(LAYERS), y=ratio.tolist(),
        name=model, mode='lines+markers',
        line=dict(color=COLORS[model], dash=DASH[model], width=2),
    ))

fig.update_layout(
    title='Accent-Phoneme Entanglement Index by Layer',
    height=380,
    legend=dict(orientation='h', yanchor='bottom', y=1.05, xanchor='center', x=0.5),
)
fig.update_xaxes(title_text='Layer')
fig.update_yaxes(title_text='L1 sil / |phoneme sil|')
fig.show()

In [10]:
# ── 4b. L1 vs Speaker silhouette: is accent encoded beyond speaker identity? ──
# If L1 sil >> speaker sil: model encodes group-level accent, not just individual speakers.
# If L1 sil ≈ speaker sil: accent decodability is just a proxy for speaker identity.
display(HTML('<h4>L1 vs Speaker Silhouette — Is Accent Group-Level or Just Speaker Identity?</h4>'))
display(HTML('<p style="color:#888">If L1 ≈ speaker: accent is just speaker ID. If L1 > speaker: genuine group-level accent structure.</p>'))

for model in MODELS:
    l1_s   = df[(df['model']==model) & (df['label_type']=='l1')].sort_values('layer')
    spk_s  = df[(df['model']==model) & (df['label_type']=='speaker')].sort_values('layer')
    diff   = l1_s['silhouette'].values - spk_s['silhouette'].values
    print(f'{model}: L1-Speaker silhouette delta by layer:')
    for li, d in zip(LAYERS, diff):
        bar = '█' * int(abs(d) * 2000)
        sign = '+' if d >= 0 else '-'
        print(f'  Layer {li}: {sign}{abs(d):.4f}  {bar}')
    print()

fig = go.Figure()
for model in MODELS:
    l1_s  = df[(df['model']==model) & (df['label_type']=='l1')].sort_values('layer')
    spk_s = df[(df['model']==model) & (df['label_type']=='speaker')].sort_values('layer')
    fig.add_trace(go.Scatter(
        x=list(LAYERS),
        y=(l1_s['silhouette'].values - spk_s['silhouette'].values).tolist(),
        name=model, mode='lines+markers',
        line=dict(color=COLORS[model], dash=DASH[model], width=2),
    ))

fig.add_hline(y=0, line_dash='dot', line_color='gray')
fig.update_layout(
    title='L1 − Speaker Silhouette Delta by Layer',
    height=380,
    legend=dict(orientation='h', yanchor='bottom', y=1.05, xanchor='center', x=0.5),
)
fig.update_xaxes(title_text='Layer')
fig.update_yaxes(title_text='Δ Silhouette (L1 − Speaker)')
fig.show()

baseline: L1-Speaker silhouette delta by layer:
  Layer 0: +0.0028  █████
  Layer 1: -0.0021  ████
  Layer 2: +0.0001  
  Layer 3: -0.0029  █████
  Layer 4: -0.0007  █
  Layer 5: +0.0004  
  Layer 6: +0.0013  ██

lora: L1-Speaker silhouette delta by layer:
  Layer 0: +0.0045  ████████
  Layer 1: -0.0001  
  Layer 2: +0.0021  ████
  Layer 3: -0.0015  ███
  Layer 4: +0.0001  
  Layer 5: +0.0019  ███
  Layer 6: +0.0005  █



In [13]:
# ── 4c. Focus phone UMAPs if available ────────────────────────────────────────
display(HTML('<h4>Focus Phones: /θ/, /ð/, /v/, /f/, /s/, /z/ — Phoneme + L1 Colouring</h4>'))
display(HTML('<p style="color:#888">These are the phones most affected by L2 transfer errors. '
             'If same-phone tokens cluster by L1, accent is encoded inside phoneme representations.</p>'))

for layer in [0, 3, 6]:
    display(HTML(f'<b>Layer {layer}</b>'))
    row_imgs = []
    for model in MODELS:
        for plot_type in ['focus_phone', 'focus_l1']:
            path = PLOTS_DIR / model / f'umap_{model}_layer{layer}_{plot_type}.png'
            if path.exists():
                img = widgets.Image(value=open(path,'rb').read(), format='png',
                                    layout=widgets.Layout(width='24%'))
                box = widgets.VBox([
                    widgets.HTML(f'<b style="font-size:11px">{model} | {plot_type}</b>'), img
                ], layout=widgets.Layout(align_items='center', width='24%'))
                row_imgs.append(box)
    if row_imgs:
        display(widgets.HBox(row_imgs,
            layout=widgets.Layout(justify_content='space-around')))
    display(HTML('<hr style="border:0.5px solid #eee"/>'))

In [14]:
# ── 4d. Summary radar / spider chart ─────────────────────────────────────────
# Normalise metrics 0-1 across layers for a high-level 'best layer' view
display(HTML('<h4>Best Layer Summary — Normalised Metric Radar</h4>'))

import plotly.graph_objects as go

radar_metrics = ['silhouette', 'davies_bouldin', 'calinski_harabasz', 'wb_ratio']
radar_labels  = ['Silhouette ↑', 'DB ↓ (inv)', 'CH ↑', 'W/B ↓ (inv)']

fig = go.Figure()
for model in MODELS:
    for label_type in ['phoneme', 'l1']:
        sub = df[(df['model']==model) & (df['label_type']==label_type)]
        # Take best layer value for each metric
        vals = []
        for metric in radar_metrics:
            if metric in ['davies_bouldin', 'wb_ratio']:
                vals.append(float(sub[metric].min()))  # lower is better
            else:
                vals.append(float(sub[metric].max()))  # higher is better
        fig.add_trace(go.Scatterpolar(
            r=vals + [vals[0]],
            theta=radar_labels + [radar_labels[0]],
            name=f'{model}/{label_type}',
            fill='toself', opacity=0.3,
        ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True)),
    title='Best-Layer Metric Summary (raw values — axes not comparable across metrics)',
    height=500,
    legend=dict(orientation='h', yanchor='bottom', y=-0.2, xanchor='center', x=0.5),
)
fig.show()